In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType, DateType
import pyspark.sql.functions as F

# 1. Inicializar sesión de Spark
spark = SparkSession.builder.appName("BronzeToSilverSales").getOrCreate()

# --- NUEVO: Configuración del parámetro/widget para Azure Data Factory ---
# Declara el widget receptor. El nombre "p_fecha_proceso" debe coincidir con el parámetro en ADF.
dbutils.widgets.text("p_fecha_proceso", "2026-09-21", "Fecha inyectada por ADF")

# Capturar el valor enviado por ADF (ej: "2026-09-21")
fecha_adf = dbutils.widgets.get("p_fecha_proceso")

# Formatear a formato compacto para el nombre de archivo (ej: "20260921")
fecha_compacta = fecha_adf.replace("-", "")
# ------------------------------------------------------------------------

# 2. Definir el esquema estricto (StructType) para el archivo de entrada
schema_entrada = StructType([
    StructField("ID_Venta", StringType(), True),
    StructField("Fecha_orden", StringType(), True),  
    StructField("Producto_Categoria", StringType(), True),
    StructField("Cantidad", IntegerType(), True),
    StructField("Precio_Unitario", DoubleType(), True),
    StructField("Total", DoubleType(), True),
    StructField("Region", StringType(), True),
    StructField("Cliente", StringType(), True),
    StructField("Pais", StringType(), True),
    StructField("Ciudad", StringType(), True),
    StructField("Es_Venta", BooleanType(), True),
    StructField("Cancelada", BooleanType(), True)
])

# 3. Definir rutas en Unity Catalog (Modificado ruta_origen para archivo específico)
nombre_archivo = f"SalesSampleTest_{fecha_compacta}.csv"
ruta_origen = f"/Volumes/workspace/default/csvfiles/sales/{nombre_archivo}"

ruta_cuarentena = "/Volumes/workspace/default/csvfiles/quarantine/sales_bad_records"
ruta_destino_silver = "/Volumes/workspace/default/csvfiles/silver/sales_clean"

print(f"--- INICIO DE PROCESAMIENTO ORQUESTADO POR ADF ---")
print(f"Fecha recibida de ADF: {fecha_adf}")
print(f"Buscando archivo diario: {ruta_origen}")

# 4. Leer los datos aplicando control de calidad con badRecordsPath y validando existencia
try:
    df_bronze = spark.read.format("csv") \
        .schema(schema_entrada) \
        .option("header", "true") \
        .option("delimiter", ",") \
        .option("badRecordsPath", ruta_cuarentena) \
        .load(ruta_origen)
        
    print(f"¡Éxito! Archivo diario localizado. Registros iniciales leídos: {df_bronze.count()}")

except Exception as e:
    # Si el archivo no existe, lanza un error para detener y notificar el fallo en ADF
    raise ValueError(f"Fallo en pipeline: No se encontró el archivo '{nombre_archivo}' esperado por ADF en la ruta del volumen.")

# 5. Fase de Limpieza y Transformación (Silver)
df_silver = df_bronze \
    .filter(F.col("ID_Venta").isNotNull()) \
    .dropDuplicates(["ID_Venta"]) \
    .withColumn("Fecha_orden_Limpia", F.try_to_date(F.col("Fecha_orden"), "yyyy-MM-dd")) \
    .withColumn("Anio", F.year(F.col("Fecha_orden_Limpia"))) \
    .withColumn("Mes", F.format_string("%02d", F.month(F.col("Fecha_orden_Limpia")))) \
    .drop("Fecha_orden") \
    .withColumnRenamed("Fecha_orden_Limpia", "Fecha_orden")

# Reordenar columnas para dejar las columnas de partición al final
columnas_ordenadas = [
    "ID_Venta", "Fecha_orden", "Producto_Categoria", "Cantidad", 
    "Precio_Unitario", "Total", "Region", "Cliente", "Pais", 
    "Ciudad", "Es_Venta", "Cancelada", "Anio", "Mes"
]
df_silver = df_silver.select(columnas_ordenadas)

# 6. Guardar en formato Delta Parquet particionado por Año y Mes
df_silver.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("Anio", "Mes") \
    .save(ruta_destino_silver)

print(f"Transformación Silver completada con éxito.")
print(f"Datos limpios guardados en Delta: {ruta_destino_silver}")
print(f"Registros corruptos (si hubo) aislados en: {ruta_cuarentena}")


In [0]:
# Leer el directorio Delta como un DataFrame
df_silver_resultado = spark.read.format("delta").load("/Volumes/workspace/default/csvfiles/silver/sales_clean")

# Visualizar con la interfaz gráfica interactiva de Databricks
display(df_silver_resultado)